In [ ]:
"""
06_pseudotime_reconciliation.py
Trajectory gate: do Velorama's DPT axis and Flufftail's Monocle3 axis describe the
SAME hepatocyte->cholangiocyte trajectory? Both are computed from ONE shared
hepatocyte root, then compared. Agreement here is what licenses comparing the two
methods' GRNs in step 07.

Reads : prepared .h5ad (step 00; needs Harmony UMAP + X_pca)
Writes: 99_results/pseudotime_comparison/*  (summary CSV + comparison plots)
"""
import os
from gribben_config import *          # PREPARED_H5AD, PSEUDO_DIR, DATASET_PREPARED
import numpy as np, pandas as pd, scipy.sparse as sp
import scanpy as sc, anndata as ad, matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr

# rpy2 bridge to run Monocle3 (optional: if unavailable we still emit DPT + a note)
try:
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri, pandas2ri
    from rpy2.robjects.conversion import localconverter
    RPY2 = True
except Exception as e:
    RPY2 = False; print("rpy2/Monocle3 unavailable:", e)

OUTPUT_DIR = PSEUDO_DIR
COLORS = {"Hepatocytes": "#E87C4C", "Cholangiocytes": "#4C9BE8"}

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100


In [ ]:
# === STEP 1: load + guarantee a log-normalised X (for marker scoring) ===
adata = ad.read_h5ad(PREPARED_H5AD)
if adata.X is None or (sp.issparse(adata.X) and adata.X.nnz == 0):
    adata.X = adata.layers["counts"].copy()
    sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)

In [ ]:
# --- VIZ: dataset overview before computing either trajectory ---
plt.figure(figsize=(6,5))
cols = adata.obs['cell.annotation'].map(lambda c: COLORS.get(c, '#aaa')).values
plt.scatter(*adata.obsm['X_umap'].T, c=cols, s=5, alpha=.6)
plt.title(f'Input to reconciliation: {adata.n_obs} cells'); plt.xlabel('UMAP1'); plt.ylabel('UMAP2')
plt.tight_layout(); plt.show()


In [ ]:
# === STEP 2: one shared root = the most hepatocyte-like hepatocyte ===
HEP_MARKERS = ["ALB","APOB","TTR","TF","HNF4A","CYP2E1","ASS1","SERPINA1","CPS1","APOA1"]
def pick_root(a, key="hep_score"):
    hep = np.where((a.obs["cell.annotation"] == "Hepatocytes").values)[0]
    mk = [g for g in HEP_MARKERS if g in a.var_names]
    sc.tl.score_genes(a, mk, score_name=key, random_state=0)
    return int(hep[np.argmax(a.obs[key].values[hep])])

In [ ]:
# === STEP 3: DPT and Monocle3 from the same root ===
def _norm(x):
    x = np.where(np.isfinite(x), x, np.nan); lo, hi = np.nanmin(x), np.nanmax(x)
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)

def dpt_from(a, iroot):
    sc.pp.neighbors(a, n_neighbors=15, use_rep="X_pca"); sc.tl.diffmap(a)
    a.uns["iroot"] = int(iroot); sc.tl.dpt(a)
    return _norm(a.obs["dpt_pseudotime"].values.astype(float))

def monocle3_from(a, root_name):
    if not RPY2: return np.full(a.n_obs, np.nan)
    ro.r('suppressPackageStartupMessages({library(monocle3);library(SingleCellExperiment);library(Matrix)})')
    names = list(map(str, a.obs_names))
    with localconverter(ro.default_converter + numpy2ri.converter):
        ro.globalenv["counts_mat"] = np.ones((1, a.n_obs), dtype=np.float32)   # dummy; counts unused
        ro.globalenv["pca_embed"]  = a.obsm["X_pca"].astype(np.float64)
        ro.globalenv["umap_embed"] = a.obsm["X_umap"].astype(np.float64)
    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["cell_meta"] = a.obs[["cell.annotation"]].set_axis(names)
    ro.globalenv["cell_names"] = ro.StrVector(names)
    ro.globalenv["root_name"]  = ro.StrVector([str(root_name)])
    ro.r('''
      colnames(counts_mat) <- cell_names; rownames(counts_mat) <- "DUMMY"
      rownames(cell_meta)  <- cell_names
      cds <- new_cell_data_set(counts_mat, cell_metadata = cell_meta,
              gene_metadata = data.frame(gene_short_name="DUMMY", row.names="DUMMY"))
      reducedDims(cds)[["PCA"]] <- pca_embed; reducedDims(cds)[["UMAP"]] <- umap_embed
      set.seed(42)
      cds <- cluster_cells(cds, reduction_method="UMAP", verbose=FALSE)
      cds <- learn_graph(cds, use_partition=FALSE, verbose=FALSE)
      cds <- order_cells(cds, root_cells = root_name[1])          # SHARED root
      monocle_pt <- pseudotime(cds)''')
    return _norm(np.array(ro.globalenv["monocle_pt"], dtype=float))

In [ ]:
# === STEP 4: align orientation + score agreement ===
def align_score(dpt, m3, root_pos):
    ok = np.isfinite(dpt) & np.isfinite(m3)
    flip = (spearmanr(dpt[ok], m3[ok])[0] < -0.1) or (np.isfinite(m3[root_pos]) and m3[root_pos] > 0.5)
    m3a = (1 - m3) if flip else m3
    return m3a, dict(spearman=spearmanr(dpt[ok], m3a[ok])[0],
                     pearson=pearsonr(dpt[ok], m3a[ok])[0], flipped=flip, n=int(ok.sum()))

def compare(a, dpt, m3a, root_pos, title, fname):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
    cols = a.obs["cell.annotation"].map(lambda c: COLORS.get(c, "#aaa")).values
    ax[0].scatter(dpt, m3a, c=cols, s=6, alpha=.5); ax[0].plot([0,1],[0,1],"k--",lw=1)
    ax[0].set_xlabel("DPT"); ax[0].set_ylabel("Monocle3"); ax[0].set_title("agreement")
    for k, (a_, v, t) in enumerate([(ax[1], dpt, "DPT"), (ax[2], m3a, "Monocle3")]):
        s = a_.scatter(*a.obsm["X_umap"].T, c=v, cmap="viridis", s=5, alpha=.7)
        a_.scatter(*a.obsm["X_umap"][root_pos], c="red", s=160, marker="*"); a_.set_title(t)
        plt.colorbar(s, ax=a_, fraction=.046)
    plt.suptitle(title, weight="bold"); plt.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches="tight"); plt.close(fig)

In [ ]:
# === STEP 5: comparison 1 — all cells (the global hep->chol axis) ===
r_all = pick_root(adata)
dpt_all = dpt_from(adata.copy(), r_all)
m3_all, s_all = align_score(dpt_all, monocle3_from(adata.copy(), str(adata.obs_names[r_all])), r_all)
compare(adata, dpt_all, m3_all, r_all, "All cells (shared root)",
        os.path.join(OUTPUT_DIR, "cmp_allcells.png"))
print("ALL CELLS:", s_all)

In [ ]:
# === STEP 6: comparison 2 — hepatocytes only (fine within-lineage structure) ===
ad_h = adata[adata.obs["cell.annotation"] == "Hepatocytes"].copy()
r_h = pick_root(ad_h)
dpt_h = dpt_from(ad_h.copy(), r_h)
m3_h, s_h = align_score(dpt_h, monocle3_from(ad_h.copy(), str(ad_h.obs_names[r_h])), r_h)
compare(ad_h, dpt_h, m3_h, r_h, "Hepatocytes only (shared root)",
        os.path.join(OUTPUT_DIR, "cmp_hepatocytes.png"))
print("HEPATOCYTES ONLY:", s_h)

In [ ]:
# === STEP 7: summary ===
pd.DataFrame([{"condition": "all cells", **s_all},
              {"condition": "hepatocytes only", **s_h}]).set_index("condition") \
  .to_csv(os.path.join(OUTPUT_DIR, "dpt_vs_monocle3_summary.csv"))
print("High Spearman on 'all cells' => methods share the hep->chol axis (step 07 comparison is licensed).")

## NEXT STEPS / GAPS
## - Monocle3's learn_graph(use_partition=FALSE) forces one graph; verify that matches
##   how Flufftail actually ran Monocle3, else the axes are not strictly comparable.
## - This compares a re-run of Monocle3. To compare against Flufftail's *actual*
##   pseudotime, load flufftail_pseudotime.csv and correlate directly (step 07 does this).
## - If 'all cells' agrees but 'hepatocytes only' does not, DPT (diffusion) and Monocle3
##   (graph geodesic) diverge on fine structure — expected, and fine for stage-level GRNs.

In [ ]:
# --- VIZ: agreement summary across both comparisons ---
summ = pd.DataFrame([{'condition':'all cells', **s_all}, {'condition':'hepatocytes only', **s_h}])
plt.figure(figsize=(5,3.5))
plt.bar(summ.condition, summ.spearman, color=['#3576b0','#e88035'])
plt.axhline(0.7, color='green', linestyle='--', alpha=.6, label='good agreement')
plt.ylabel('Spearman(DPT, Monocle3)'); plt.title('DPT vs Monocle3 agreement'); plt.legend()
plt.tight_layout(); plt.show()
